In [ ]:
print("CodeAlpha Car Price Prediction Project")
print("Student Name: Srishti Pandey")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

print("Libraries imported successfully!")

In [ ]:
import os
import sys
from pathlib import Path

dataset_path = os.getenv("CAR_DATASET_PATH")

if dataset_path:
    csv_path = Path(dataset_path)
    if not csv_path.is_file():
        raise FileNotFoundError(f"CAR_DATASET_PATH does not exist: {csv_path}")
    uploaded_filename = str(csv_path)
    print(f"Using dataset from CAR_DATASET_PATH: {uploaded_filename}")
elif "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Please upload the used-car CSV file.")
    uploaded_filename = next(iter(uploaded))
    print(f"Uploaded file: {uploaded_filename}")
else:
    import kagglehub

    dataset_dir = Path(
        kagglehub.dataset_download("taeefnajib/used-car-price-prediction-dataset")
    )
    matches = list(dataset_dir.rglob("used_cars.csv"))
    if not matches:
        raise FileNotFoundError(
            "Kaggle download completed, but used_cars.csv was not found. "
            "Set CAR_DATASET_PATH to a valid CSV file."
        )
    uploaded_filename = str(matches[0])
    print(f"Downloaded public Kaggle dataset: {uploaded_filename}")


In [ ]:
df = pd.read_csv(uploaded_filename)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

In [ ]:
print("Dataset Information:")
df.info()

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
# Keep missing values for now. Imputation is fitted later on the training data
# inside the scikit-learn preprocessing pipeline to avoid data leakage.
print("Missing values retained for train-only preprocessing.")
print(df.isnull().sum())

In [ ]:
print("Column Data Types:")
print(df.dtypes)

In [ ]:
# Convert price into numeric
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["price"] = pd.to_numeric(df["price"], errors="coerce")


# Dataset uses "milage" spelling
df["milage"] = (
    df["milage"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+\.?\d*)")[0]
)

df["milage"] = pd.to_numeric(df["milage"], errors="coerce")

print("Price and milage converted successfully!")
print(df[["price", "milage"]].dtypes)

In [ ]:
print("Price Statistics:")
print(df["price"].describe())

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["price"], bins=30)

plt.xlabel("Car Price")
plt.ylabel("Number of Cars")
plt.title("Distribution of Car Prices")

plt.show()

In [ ]:
brand_price = (
    df.groupby("brand")["price"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

print("Top 15 Brands by Average Price:")
print(brand_price)

In [ ]:
plt.figure(figsize=(10,6))

brand_price.sort_values().plot(kind="barh")

plt.xlabel("Average Price")
plt.ylabel("Brand")
plt.title("Top 15 Brands by Average Car Price")

plt.show()

In [ ]:
# Keep a fixed reference year so the engineered feature is reproducible.
REFERENCE_YEAR = 2026
df["model_year"] = pd.to_numeric(df["model_year"], errors="coerce")
df["car_age"] = (REFERENCE_YEAR - df["model_year"]).clip(lower=0)

print("Car age feature created successfully!")
print(df[["model_year", "car_age"]].head())

In [ ]:
# Reuse the tested cleaning helper so notebook and unit-test behaviour stay aligned.
from src.car_data import clean_cars

rows_before = len(df)
df = clean_cars(df, reference_year=REFERENCE_YEAR)
print(f"Removed {rows_before - len(df)} rows with invalid price or model year.")
print(f"Cleaned dataset shape: {df.shape}")


In [ ]:
# Select one representation of vehicle age. model_year and car_age are redundant.
features = [
    "brand",
    "model",
    "car_age",
    "mileage",
    "fuel_type",
    "engine",
    "transmission",
    "ext_col",
    "int_col",
    "accident",
    "clean_title",
]

# Create X and y
X = df[features]
y = df["price"]

# Stratify by target-price quantiles so the holdout contains a representative
# spread of low-, mid-, and high-priced vehicles. Bins are used only for
# splitting, never as model features.
price_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=price_bins,
)

print("Features selected successfully!")
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Target-price strata represented in train and holdout.")

In [ ]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numeric_features)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Imputers are fitted only on X_train when the model pipeline is trained.
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, categorical_features),
        ("numeric", numeric_transformer, numeric_features)
    ]
)

print("Train-only preprocessing setup completed successfully!")
print("Categorical features:", categorical_features)
print("Numerical features:", numeric_features)

## Cross-validation and baseline comparison

A single train/test split can give an unstable estimate. This section compares a median-price baseline, Linear Regression, Random Forest and Extra Trees using shuffled five-fold cross-validation. Every fold fits its own imputation and encoding pipeline on the training partition, preventing preprocessing leakage.

MAE and RMSE are reported in the dataset's price units. R² measures explained variance. Compare mean scores and fold-to-fold standard deviations. A strong result should outperform the median baseline consistently, not only on one split.


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import StratifiedKFold, cross_validate

# Keep the 20% test set untouched during model selection.
# Target-price bins are used only to distribute the price range across folds.
train_price_bins = pd.qcut(y_train, q=10, labels=False, duplicates="drop")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

def make_tree_pipeline(model):
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

cv_models = {
    "Median baseline": make_tree_pipeline(DummyRegressor(strategy="median")),
    "Linear Regression": make_tree_pipeline(LinearRegression()),
    "Random Forest": make_tree_pipeline(RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1,
        min_samples_leaf=2,
    )),
    "Extra Trees": make_tree_pipeline(ExtraTreesRegressor(
        n_estimators=150, random_state=42, n_jobs=-1,
        min_samples_leaf=2, max_features=0.8,
    )),
    "Random Forest (log target)": TransformedTargetRegressor(
        regressor=make_tree_pipeline(RandomForestRegressor(
            n_estimators=100, random_state=42, n_jobs=-1,
            min_samples_leaf=2,
        )),
        func=np.log1p,
        inverse_func=np.expm1,
    ),
    "Extra Trees (log target)": TransformedTargetRegressor(
        regressor=make_tree_pipeline(ExtraTreesRegressor(
            n_estimators=150, random_state=42, n_jobs=-1,
            min_samples_leaf=2, max_features=0.8,
        )),
        func=np.log1p,
        inverse_func=np.expm1,
    ),
}

cv_rows = []
for model_name, estimator in cv_models.items():
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv.split(X_train, train_price_bins),
        scoring=scoring, n_jobs=1, return_train_score=False,
    )
    cv_rows.append({
        "Model": model_name,
        "Mean CV MAE": -scores["test_mae"].mean(),
        "MAE Std": scores["test_mae"].std(),
        "Mean CV RMSE": -scores["test_rmse"].mean(),
        "RMSE Std": scores["test_rmse"].std(),
        "Mean CV R2": scores["test_r2"].mean(),
        "R2 Std": scores["test_r2"].std(),
    })

cv_comparison = pd.DataFrame(cv_rows).sort_values("Mean CV MAE")
display(cv_comparison.round(3))
print("Cross-validation uses only the training split and target-stratified folds.")
print("The untouched 20% holdout set is reserved for one final evaluation.")

best_model_name = cv_comparison.iloc[0]["Model"]
best_model = clone(cv_models[best_model_name])
best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

best_mae = mean_absolute_error(y_test, best_pred)
best_rmse = np.sqrt(mean_squared_error(y_test, best_pred))
best_r2 = r2_score(y_test, best_pred)

In [ ]:
# Evaluate the selected model on the untouched holdout set.
plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_pred, alpha=0.5)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title(f"Actual vs Predicted Car Prices - {best_model_name}")
plt.tight_layout()
plt.show()

residuals = y_test.to_numpy() - best_pred
plt.figure(figsize=(8, 5))
plt.scatter(best_pred, residuals, alpha=0.5)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title(f"Holdout Residual Analysis - {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
# Inspect holdout predictions from the selected model, not a different baseline model.
sample_result = X_test.copy()
sample_result["Actual Price"] = y_test.values
sample_result["Predicted Price"] = best_pred
sample_result["Absolute Error"] = np.abs(y_test.values - best_pred)
sample_result["Absolute Percentage Error (%)"] = (
    sample_result["Absolute Error"] / y_test.values * 100
).round(2)
display(sample_result.head(10))

In [ ]:
# Diagnose where holdout errors concentrate across the actual-price distribution.
holdout_diagnostics = pd.DataFrame({
    "Actual Price": np.asarray(y_test, dtype=float),
    "Predicted Price": np.asarray(best_pred, dtype=float),
})
holdout_diagnostics["Absolute Error"] = np.abs(
    holdout_diagnostics["Actual Price"] - holdout_diagnostics["Predicted Price"]
)
holdout_diagnostics["Absolute Percentage Error (%)"] = (
    holdout_diagnostics["Absolute Error"]
    / holdout_diagnostics["Actual Price"].replace(0, np.nan)
    * 100
)

# Quantile bins are created only for post-hoc error inspection, not model fitting.
number_of_bands = min(5, int(holdout_diagnostics["Actual Price"].nunique()))
if number_of_bands < 2:
    raise ValueError("Not enough distinct holdout prices for price-band diagnostics.")

holdout_diagnostics["Actual Price Band"] = pd.qcut(
    holdout_diagnostics["Actual Price"],
    q=number_of_bands,
    duplicates="drop",
)

band_rows = []
for band, group in holdout_diagnostics.groupby("Actual Price Band", observed=True):
    actual = group["Actual Price"].to_numpy(dtype=float)
    predicted = group["Predicted Price"].to_numpy(dtype=float)
    absolute_error = np.abs(actual - predicted)
    band_rows.append({
        "Actual Price Band": str(band),
        "Listings": int(len(group)),
        "Median Actual Price": float(np.median(actual)),
        "MAE": float(np.mean(absolute_error)),
        "RMSE": float(np.sqrt(np.mean((actual - predicted) ** 2))),
        "WAPE (%)": float(100 * absolute_error.sum() / np.abs(actual).sum()),
        "Mean Absolute Percentage Error (%)": float(
            group["Absolute Percentage Error (%)"].mean()
        ),
    })

price_band_diagnostics = pd.DataFrame(band_rows)
print("Holdout error by actual-price band (post-hoc diagnostic only):")
display(price_band_diagnostics.round(2))
print(
    "These groups diagnose the untouched holdout errors. "
    "They are not used to train or select the model."
)


In [ ]:
# Permutation importance for the selected model on the holdout set.
# This is post-hoc interpretation. It is not used for model selection.
importance_result = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=10,
    random_state=42,
    n_jobs=1,
)
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance (MAE increase)": importance_result.importances_mean,
    "Importance Std": importance_result.importances_std,
}).sort_values("Importance (MAE increase)", ascending=False)

print("Permutation importance for selected model:", best_model_name)
display(feature_importance)

top_features = feature_importance.head(15).sort_values("Importance (MAE increase)")
plt.figure(figsize=(10, 6))
plt.barh(
    top_features["Feature"],
    top_features["Importance (MAE increase)"],
    xerr=top_features["Importance Std"],
)
plt.xlabel("Increase in holdout MAE after permutation")
plt.ylabel("Original input feature")
plt.title(f"Selected Model Feature Importance - {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
print("========== CAR PRICE PREDICTION PROJECT ==========")
print("Dataset Size:", df.shape)
print("Models evaluated: Median baseline, Linear Regression, Random Forest, Extra Trees, and log-target tree models")
print("Best model selected by training-only CV MAE:", best_model_name)
print("Training-only CV MAE:", round(cv_comparison.iloc[0]["Mean CV MAE"], 2))
print("Training-only CV RMSE:", round(cv_comparison.iloc[0]["Mean CV RMSE"], 2))
print("Training-only CV R2:", round(cv_comparison.iloc[0]["Mean CV R2"], 3))
print("Untouched holdout MAE:", round(best_mae, 2))
print("Untouched holdout RMSE:", round(best_rmse, 2))
print("Untouched holdout R2:", round(best_r2, 4))
print("These results are exploratory and do not establish production readiness.")

## Conclusion

This project compares a median-price baseline with Linear Regression, Random Forest and Extra Trees. It uses train-only preprocessing pipelines and five-fold cross-validation to reduce dependence on a single split.

Model quality must be judged from MAE, RMSE, R² and fold-to-fold variation together. A positive R² alone does not make the model reliable. The current dataset is small and contains a wide range of vehicle prices, so extreme but valid listings can produce large RMSE values.

Do not use the current model for real purchasing or valuation decisions. Further work should include careful outlier investigation, hyperparameter tuning, a log-target experiment, and evaluation on a genuinely independent or later-period dataset. Do not remove valid luxury vehicles only to improve scores.
